# 12 Full CV Preprocess

目的: downloaded / local source video から `candidate_segments_v1`, `clips_v1`, `debug_overlays_v1` を作ります。

これは full CV pipeline の最初の実装です。現段階では軽量 heuristic による video probe / motion peak contact estimate / contact-centered trim を行います。YOLO、pose、bat、plate detector は同じ contract の上に後で差し替えます。

出力:

- `/content/drive/MyDrive/baseball_vision/clips/mlb_2024_2026_full_v1/candidate_segments_v1.parquet`
- `/content/drive/MyDrive/baseball_vision/clips/mlb_2024_2026_full_v1/clips_v1.parquet`
- `/content/drive/MyDrive/baseball_vision/debug/mlb_2024_2026_full_v1/debug_overlays_v1.parquet`


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec('cv2') is None:
    print('Installing opencv-python-headless for video probe / contact estimate...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'])
else:
    print('cv2 is available')

ffmpeg_check = subprocess.run(['which', 'ffmpeg'], capture_output=True, text=True)
print('ffmpeg =', ffmpeg_check.stdout.strip() or 'not found')

from sport_pipeline.runtime import summarize_runtime_device

device = summarize_runtime_device(prefer_gpu=True, require_gpu=False)
device


## Run Preprocessing

`EXTRACT_CLIPS=True` にすると ffmpeg で contact-centered mp4 を作ります。最初は `MAX_SOURCES=5` 程度で確認してください。

In [ ]:
from sport_pipeline.pipeline.preprocess.full_cv import run_full_cv_preprocess

CV_SETTINGS = stage_settings(RUN_PROFILE, 'full_cv')
MAX_SOURCES = CV_SETTINGS.get('max_sources')
EXTRACT_CLIPS = bool(CV_SETTINGS.get('extract_clips', True))
REQUIRE_NON_EMPTY_CLIPS = bool(CV_SETTINGS.get('require_non_empty', True))
REQUIRE_EVENT_LEVEL_MATCH = bool(CV_SETTINGS.get('require_event_level_match', True))
MIN_VIDEO_MATCH_CONFIDENCE = float(CV_SETTINGS.get('min_video_match_confidence', 0.45))
CONTACT_SEARCH_START_SEC = float(CV_SETTINGS.get('contact_search_start_sec', 0.5))
CONTACT_SEARCH_END_SEC = CV_SETTINGS.get('contact_search_end_sec', 8.0)
CLIP_VERSION = str(CV_SETTINGS.get('clip_version', 'pre_contact_long'))
RESUME_CV = bool(CV_SETTINGS.get('resume', True))
CHECKPOINT_EVERY_SOURCES = int(CV_SETTINGS.get('checkpoint_every_sources', 10))
CACHE_INPUTS = bool(CV_SETTINGS.get('cache_inputs', False))
CACHE_OUTPUTS = bool(CV_SETTINGS.get('cache_outputs', False))
CACHE_MIN_FREE_DISK_GB = float(CV_SETTINGS.get('cache_min_free_disk_gb', 20))
CACHE_MAX_FILE_MB = CV_SETTINGS.get('cache_max_file_mb')
NUM_WORKERS = int(CV_SETTINGS.get('num_workers', 1))
PROGRESS_LOG_EVERY_SOURCES = int(CV_SETTINGS.get('progress_log_every_sources', 0))

print('full_cv cache_inputs =', CACHE_INPUTS, 'cache_outputs =', CACHE_OUTPUTS, 'num_workers =', NUM_WORKERS)

summary = run_full_cv_preprocess(
    base_dir=BASE_DIR,
    run_id=RUN_ID,
    max_sources=MAX_SOURCES,
    extract_clips=EXTRACT_CLIPS,
    require_non_empty=REQUIRE_NON_EMPTY_CLIPS,
    require_event_level_match=REQUIRE_EVENT_LEVEL_MATCH,
    min_video_match_confidence=MIN_VIDEO_MATCH_CONFIDENCE,
    contact_search_start_sec=CONTACT_SEARCH_START_SEC,
    contact_search_end_sec=CONTACT_SEARCH_END_SEC,
    clip_version=CLIP_VERSION,
    resume=RESUME_CV,
    checkpoint_every_sources=CHECKPOINT_EVERY_SOURCES,
    cache_dir=CACHE_DIR,
    cache_inputs=CACHE_INPUTS,
    cache_outputs=CACHE_OUTPUTS,
    cache_min_free_disk_gb=CACHE_MIN_FREE_DISK_GB,
    cache_max_file_mb=CACHE_MAX_FILE_MB,
    num_workers=NUM_WORKERS,
    progress_log_every_sources=PROGRESS_LOG_EVERY_SOURCES,
)

summary


In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.cv.diagnostics import clip_quality_diagnostics, format_clip_quality_diagnostics
from sport_pipeline.io import read_table

expected = [
    f'clips/{RUN_ID}/candidate_segments_v1.parquet',
    f'clips/{RUN_ID}/clips_v1.parquet',
    f'debug/{RUN_ID}/debug_overlays_v1.parquet',
    f'reports/preflight/full_cv_preprocess_{RUN_ID}.json',
    f'reports/preflight/full_cv_preprocess_{RUN_ID}_progress.json',
]
check = check_artifacts(BASE_DIR, expected)
print(check)

clips_path = BASE_DIR / 'clips' / RUN_ID / 'clips_v1.parquet'
clip_rows = read_table(clips_path) if clips_path.exists() else []
diagnostics = clip_quality_diagnostics(clip_rows)
print(format_clip_quality_diagnostics(diagnostics))
min_clean_clips = threshold(RUN_PROFILE, 'min_clean_clips', 0)
print('min_clean_clips threshold =', min_clean_clips)
if not clip_rows:
    raise RuntimeError('12 produced 0 clips. Do not continue to 13/17/18. Go back to 11 and get downloaded video rows first.')
if diagnostics['clean_trainable_clips'] < min_clean_clips:
    raise RuntimeError(
        '12 produced too few clean trainable clips. Do not continue to 13/17/18. '
        'The diagnostics above show whether rows are failing because of unknown view, low visibility, uncertain contact, replay/cutaway, or difficult pitch. '
        'If this notebook was rerun after code changes, set full_cv.resume=False once or remove the old clips/{RUN_ID} outputs so 12 recomputes candidates.'
    )


次:

- `clips_v1.parquet` の `clip_status`, `quality_tier`, `review_reason` を確認
- clean clip が十分あれば structured sequence / video baseline へ進む
- 足りなければ `11` に戻り、video source / download を増やす
